<a href="https://colab.research.google.com/github/bil1980/atividade-telecom/blob/main/U3_atividade_telecom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade
## Normalização, escalonamento e codificação de dados categóricos
Ciência de Dados · Bacharelado em Engenharia de Software · UNIFSA


### Contexto
Uma operadora de telecomunicações deseja compreender melhor sua base de clientes. A diretoria comercial solicitou à equipe de dados dois produtos:

1. um **índice de valor do cliente**, que permita ranquear os clientes combinando tempo de relacionamento e faturamento;
2. uma **análise do cancelamento (churn)**, que indique quais características dos clientes estão associadas a uma maior taxa de cancelamento.

As métricas disponíveis estão em escalas diferentes e grande parte das informações está em formato textual. Antes de qualquer análise, é necessário preparar os dados.

### Base de dados
IBM. *Telco Customer Churn* (base de exemplo). 7.043 clientes e 21 colunas. Endereço público para carga:

https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv



### Dicionário de dados (colunas utilizadas)
| Coluna | Descrição |
|---|---|
| customerID | identificador do cliente |
| gender | sexo do titular (Male, Female) |
| Partner, Dependents | possui cônjuge; possui dependentes (Yes, No) |
| tenure | tempo de relacionamento, em meses |
| PhoneService, PaperlessBilling | possui telefone; usa fatura digital (Yes, No) |
| MultipleLines | possui múltiplas linhas (Yes, No, No phone service) |
| InternetService | tipo de internet (DSL, Fiber optic, No) |
| OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | serviços adicionais (Yes, No, No internet service) |
| Contract | tipo de contrato (Month-to-month, One year, Two year) |
| PaymentMethod | forma de pagamento (quatro categorias) |
| MonthlyCharges | valor da mensalidade |
| TotalCharges | valor total faturado desde o início do contrato |
| Churn | o cliente cancelou o serviço no último mês (Yes, No) |

---
## Parte 1 — Carga e preparação dos dados

**1.1** Carregue a base a partir do endereço público informado no início do notebook. Exiba o número de linhas e de colunas e os tipos de cada coluna.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Quantidade de linhas e colunas:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

Quantidade de linhas e colunas: (7043, 21)

Tipos de dados:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


**1.2** A coluna `TotalCharges` deveria ser numérica, mas foi lida como texto. Converta-a para numérico e identifique os registros que não puderam ser convertidos. Examine o valor de `tenure` nesses registros.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")


invalidos = df[df["TotalCharges"].isna()]

print("Quantidade de registros inválidos:", len(invalidos))


print(invalidos[["customerID", "tenure", "TotalCharges"]])

**1.3** Com base no que foi observado em 1.2, decida como tratar esses registros e implemente a decisão. Justifique a escolha considerando o significado do negócio (o que representa um cliente com esse valor de `tenure`?).

In [ ]:
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)
print("Nova dimensão:", df.shape)

**Resposta:** os 11 registros inválidos têm tenure igual a 0. São clientes que acabaram de contratar o serviço e ainda não receberam nenhuma fatura, por isso TotalCharges está vazio. Não se trata de erro de digitação, e sim de ausência de faturamento. Optei por removê-los: representam apenas cerca de 0,16% da base, não têm histórico de relacionamento nem de pagamento, nenhum deles cancelou, e portanto não ajudam nem a ranquear o valor do cliente nem a explicar o cancelamento. Imputar a média inventaria um faturamento que não existiu. Preencher com 0 seria defensável, mas puxaria artificialmente esses clientes para o fim do ranking e distorceria o escalonamento.

---
## Parte 2 — Escalonamento e índice de valor do cliente

**2.1** Apresente as estatísticas descritivas de `tenure`, `MonthlyCharges` e `TotalCharges`. Compare as amplitudes (mínimo e máximo) das três métricas.

In [ ]:
cols = ["tenure", "MonthlyCharges", "TotalCharges"]
print(df[cols].describe())
print("\nAmplitude (máx - mín):")
print(df[cols].max() - df[cols].min())

**Resposta:** as escalas são muito diferentes. tenure vai de 1 a 72 meses, MonthlyCharges de cerca de 18 a 119 e TotalCharges de cerca de 19 a 8.685. A amplitude de TotalCharges é umas 100 vezes a de tenure e umas 85 vezes a de MonthlyCharges.

**2.2** Construa um **índice bruto** somando diretamente as três métricas. Calcule a correlação do índice bruto com cada uma delas. Qual métrica determina o índice? Por quê?

In [ ]:
df["indice_bruto"] = df[cols].sum(axis=1)
print(df[cols].corrwith(df["indice_bruto"]))

**Resposta:** TotalCharges determina o índice, com correlação próxima de 1 (cerca de 0,999), enquanto tenure e MonthlyCharges têm correlação bem menor. Na soma direta, a métrica com maior variância (maior escala) domina a variação do resultado, e as outras duas quase não influenciam o ranking. O índice bruto é praticamente o próprio TotalCharges.

**2.3** Padronize as três métricas com **z-score** e construa o **índice padronizado** como a soma das três métricas padronizadas. Repita o cálculo de correlações do item 2.2.

In [ ]:
z = (df[cols] - df[cols].mean()) / df[cols].std()
df["indice_z"] = z.sum(axis=1)
print(df[cols].corrwith(df["indice_z"]))

**2.4** Liste os 10 clientes com maior índice bruto e os 10 com maior índice padronizado. Quantos clientes aparecem nas duas listas? O que mudou no perfil dos clientes mais bem posicionados?

In [ ]:
top_bruto = df.nlargest(10, "indice_bruto")
top_z = df.nlargest(10, "indice_z")

print("Top 10 bruto:")
print(top_bruto[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "indice_bruto"]])
print("\nTop 10 padronizado:")
print(top_z[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "indice_z"]])

comuns = set(top_bruto["customerID"]) & set(top_z["customerID"])
print("\nClientes nas duas listas:", len(comuns))

print("\nMédias do perfil:")
print(pd.DataFrame({
    "top_bruto": top_bruto[cols].mean(),
    "top_z": top_z[cols].mean(),
    "base": df[cols].mean()
}))

**Resposta:** o número de clientes em comum é o impresso acima (esperado: pequeno). O índice bruto escolhe os clientes de maior faturamento acumulado, em geral com tenure alto, mas sem exigir mensalidade alta. No índice padronizado sobem clientes que combinam tempo de relacionamento alto e mensalidade alta ao mesmo tempo, ou seja, o perfil fica mais equilibrado entre as três métricas.

**2.5** Aplique o **min-max** às três métricas e construa um terceiro índice. Compare a ordenação dos clientes no índice min-max com a do índice padronizado (sugestão: correlação entre os dois índices).

In [ ]:
mm = (df[cols] - df[cols].min()) / (df[cols].max() - df[cols].min())
df["indice_mm"] = mm.sum(axis=1)

print("Correlação de Pearson z vs min-max:", df["indice_z"].corr(df["indice_mm"]))
print("Correlação de Spearman z vs min-max:", df["indice_z"].corr(df["indice_mm"], method="spearman"))

**Resposta:** a correlação entre os dois índices é muito alta (próxima de 1), então a ordenação dos clientes é quase a mesma. Ambos são transformações lineares por coluna, e a diferença está só nos pesos relativos: o z-score divide pelo desvio padrão e o min-max divide pela amplitude. As colunas têm formas de distribuição semelhantes, por isso os rankings pouco divergem.

**2.6** Calcule a matriz de correlação entre `tenure`, `MonthlyCharges` e `TotalCharges`. Considerando que o faturamento total resulta, aproximadamente, do tempo de relacionamento multiplicado pela mensalidade, discuta se faz sentido manter as três métricas no índice. Proponha e implemente a versão final do índice de valor do cliente.

In [ ]:
print(df[cols].corr())

df["indice_final"] = (
    (df["tenure"] - df["tenure"].mean()) / df["tenure"].std()
    + (df["MonthlyCharges"] - df["MonthlyCharges"].mean()) / df["MonthlyCharges"].std()
)
print(df[cols].corrwith(df["indice_final"]))
print(df.nlargest(10, "indice_final")[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "indice_final"]])

**Resposta:**  TotalCharges é aproximadamente tenure x MonthlyCharges, e por isso tem correlação alta com ambas (cerca de 0,83 com tenure e 0,65 com MonthlyCharges). Mantê-la no índice conta duas vezes a mesma informação e dá peso excessivo ao eixo "tempo de relacionamento". A versão final usa apenas tenure e MonthlyCharges, padronizadas por z-score e somadas, o que representa de forma direta os dois componentes pedidos pela diretoria (tempo de relacionamento e faturamento).

---
## Parte 3 — Forma da distribuição

**3.1** Construa os histogramas das três métricas e calcule a assimetria de cada uma (`.skew()`).

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for eixo, c in zip(eixos, cols):
    eixo.hist(df[c], bins=30)
    eixo.set_title(c)
plt.show()

print(df[cols].skew())

**3.2** Aplique a transformação `np.log1p` à coluna `TotalCharges`. Compare o histograma e a assimetria antes e depois da transformação. A transformação logarítmica é adequada para esta coluna? Justifique com base nos valores obtidos.

In [ ]:
df["TotalCharges_log"] = np.log1p(df["TotalCharges"])

fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
eixos[0].hist(df["TotalCharges"], bins=30)
eixos[0].set_title("TotalCharges")
eixos[1].hist(df["TotalCharges_log"], bins=30)
eixos[1].set_title("log1p(TotalCharges)")
plt.show()

print("Assimetria antes:", df["TotalCharges"].skew())
print("Assimetria depois:", df["TotalCharges_log"].skew())

**Resposta:** TotalCharges tem assimetria positiva moderada (cerca de 0,96), com cauda à direita. Após log1p a assimetria passa para cerca de -0,75: o módulo diminui um pouco, mas o sinal se inverte, pois a transformação comprime demais a cauda direita e cria uma cauda à esquerda (muitos clientes novos com valores baixos). O histograma continua longe de simétrico. A transformação, portanto, não é claramente adequada para esta coluna: ela troca uma assimetria por outra de magnitude parecida, sem ganho relevante.

---
## Parte 4 — Codificação das variáveis categóricas

**4.1** Classifique as variáveis abaixo e indique a técnica de codificação adequada, preenchendo a tabela.

| Variável | Tipo (nominal, ordinal ou binária) | Técnica de codificação |
|---|---|---|
| gender | | |
| Partner | | |
| PhoneService | | |
| MultipleLines | | |
| InternetService | | |
| OnlineSecurity | | |
| Contract | | |
| PaymentMethod | | |
| Churn | | |

**4.2** Converta para 0 e 1 as variáveis binárias `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling` e `Churn`.

In [ ]:
df_enc = df.drop(columns=["indice_bruto", "indice_z", "indice_mm", "indice_final", "TotalCharges_log"]).copy()

df_enc["gender"] = df_enc["gender"].map({"Female": 0, "Male": 1})
for c in ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]:
    df_enc[c] = df_enc[c].map({"No": 0, "Yes": 1})

print(df_enc[["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]].head())

**4.3** As colunas de serviços adicionais (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) e a coluna `MultipleLines` possuem três valores. Analise o significado de "No internet service" e "No phone service", decida como codificar essas colunas e implemente a decisão.

In [ ]:
servicos = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
            "StreamingTV", "StreamingMovies", "MultipleLines"]
for c in servicos:
    df_enc[c] = df_enc[c].replace({"No internet service": "No", "No phone service": "No"})
    df_enc[c] = df_enc[c].map({"No": 0, "Yes": 1})

print(df_enc[servicos].head())
print(df_enc[servicos].isna().sum().sum())

**Resposta:**  "No internet service" e "No phone service" não são categorias novas: são consequências de o cliente não ter internet ou telefone, informação que já está em InternetService e PhoneService. Do ponto de vista do serviço adicional, o cliente simplesmente não o possui, equivalente a "No". Agrupar as duas categorias reduz cada coluna a uma variável binária, evita redundância e não perde informação, pois a ausência do serviço-base continua registrada nas outras colunas.

**4.4** Codifique `Contract` de forma ordinal, respeitando a ordem de duração do contrato.

In [ ]:
df_enc["Contract"] = df_enc["Contract"].map({"Month-to-month": 0, "One year": 1, "Two year": 2})
print(df_enc["Contract"].value_counts())

**4.5** Aplique one-hot encoding a `InternetService` e `PaymentMethod`. Utilize ou não o parâmetro `drop_first` e justifique a escolha.

In [ ]:
df_final = pd.get_dummies(df_enc, columns=["InternetService", "PaymentMethod"], drop_first=True, dtype=int)
print(df_final.columns.tolist())

**Resposta:**  Usei drop_first=True. Com k categorias, k colunas dummy são perfeitamente colineares (a soma é sempre 1), o que gera redundância e problemas em modelos lineares. Remover uma categoria elimina essa dependência sem perder informação, pois a categoria removida é a referência (todas as dummies em 0). Como a análise da Parte 5 usa correlações com Churn, também evita incluir uma coluna totalmente determinada pelas demais.




---
## Parte 5 — Análise do cancelamento para a diretoria comercial

**5.1** Utilizando a coluna `Churn` codificada, calcule a taxa de cancelamento (proporção de clientes que cancelaram) por tipo de contrato, por forma de pagamento e por tipo de internet.

In [ ]:
print("Por contrato:")
print(df_enc["Churn"].groupby(df["Contract"]).mean().sort_values(ascending=False))
print("\nPor forma de pagamento:")
print(df_enc["Churn"].groupby(df["PaymentMethod"]).mean().sort_values(ascending=False))
print("\nPor tipo de internet:")
print(df_enc["Churn"].groupby(df["InternetService"]).mean().sort_values(ascending=False))

**5.2** Calcule a correlação entre `Churn` e as demais variáveis codificadas e numéricas. Apresente as cinco variáveis com maior correlação em valor absoluto, indicando o sentido (positivo ou negativo) de cada uma.

In [ ]:
dados = df_final.drop(columns=["customerID"])
corr = dados.corr()["Churn"].drop("Churn")
top5 = corr.reindex(corr.abs().sort_values(ascending=False).index).head(5)
print(top5)

**5.3** Redija um parágrafo de até oito linhas dirigido à diretoria comercial, com uma recomendação para reduzir o cancelamento. A recomendação deve estar apoiada nos números obtidos em 5.1 e 5.2.

**Resposta:** o cancelamento está fortemente concentrado em clientes de contrato mensal (cerca de 43% de churn, contra cerca de 11% no contrato de um ano e 3% no de dois anos), em quem paga por cheque eletrônico (cerca de 45%, contra 15 a 19% nas demais formas) e em quem usa fibra óptica (cerca de 42%, contra 19% em DSL). As maiores correlações com Churn confirmam o quadro: o tipo de contrato e o tempo de relacionamento (tenure) têm correlação negativa, ou seja, quanto mais longo o vínculo, menor o cancelamento, enquanto fibra óptica e cheque eletrônico têm correlação positiva. Recomenda-se, portanto, oferecer descontos ou benefícios para migrar clientes do contrato mensal para contratos anuais, com atenção especial aos primeiros meses de relacionamento, e incentivar a troca do cheque eletrônico por débito ou cartão, além de investigar a qualidade do serviço de fibra óptica.

---
## Parte 6 — Questões discursivas

**6.1** Explique por que a codificação por rótulo (atribuir os números 0, 1, 2 e 3 às categorias de `PaymentMethod`) seria inadequada para a análise realizada na Parte 5.

**Resposta:** os números 0, 1, 2 e 3 impõem uma ordem e uma distância entre categorias que não existem: a forma de pagamento é nominal. A correlação com Churn, que mede relação linear, trataria "cartão" (3) como o triplo de "cheque eletrônico" (1) e dependeria da ordem arbitrária dos rótulos, podendo ocultar ou inventar associações. Como o cheque eletrônico tem churn muito acima das demais, o efeito desaparece se ele ficar no meio da escala. O one-hot preserva cada categoria separadamente.

**6.2** Descreva uma análise sobre esta base em que o escalonamento das métricas numéricas não alteraria o resultado. Justifique.

**Resposta:** a taxa de cancelamento por grupo (5.1) não muda, porque usa apenas médias da coluna Churn por categoria e não depende da escala das variáveis numéricas. Outro exemplo é o ranking de clientes por uma única métrica (como ordenar por MonthlyCharges), já que o escalonamento linear com coeficiente positivo preserva a ordem. Também não mudam árvores de decisão, que dependem apenas de comparações de limiares.

**6.3** O coeficiente de correlação entre `tenure` e `Churn` muda se `tenure` for padronizada por z-score ou escalonada por min-max? Justifique matematicamente e, em seguida, verifique no código.

In [ ]:
t = df_enc["tenure"]
churn = df_enc["Churn"]

t_z = (t - t.mean()) / t.std()
t_mm = (t - t.min()) / (t.max() - t.min())

print("Original:", t.corr(churn))
print("z-score :", t_z.corr(churn))
print("min-max :", t_mm.corr(churn))
print(np.isclose(t.corr(churn), t_z.corr(churn)), np.isclose(t.corr(churn), t_mm.corr(churn)))

**Resposta:** não muda. Tanto o z-score, z = (x - média) / s, quanto o min-max, m = (x - min) / (max - min), são transformações lineares do tipo a*x + b com a > 0. A covariância satisfaz Cov(a*X + b, Y) = a*Cov(X, Y), e o desvio padrão satisfaz s(a*X + b) = a*s(X). Na razão que define a correlação, o fator a se cancela e a constante b não tem efeito, portanto Corr(a*X + b, Y) = Corr(X, Y) para a > 0.